In [1]:
import polars as pl

In [2]:
# 1. Load your CLEAN sampled data
df = pl.read_csv("../data/total_sentence_train.csv")

In [3]:
df

PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,TOTAL_SENTENCE
i64,i64,f64,str
1925202,1650,2125.98,"""ArtzFolio Tulip Flowers Blacko…"
2673191,2755,393.7,"""Marks & Spencer Girls' Pyjama …"
2765088,7537,748.031495,"""PRIKNIK Horn Red Electric Air …"
1594019,2996,787.401574,"""ALISHAH Women's Cotton Ankle L…"
283658,6112,598.424,"""The United Empire Loyalists: A…"
…,…,…,…
2422167,3009,1181.1,"""Nike Women's As W Ny Df Swsh H…"
2766635,3413,125.984252,"""(3PCS) Goose Game Cute Cartoon…"
1987786,1574,1200.0,"""Kangroo Sweep Movement Printed…"


## CONFIG

In [ ]:

DATA_PATH        = "../data/total_sentence_train.csv"
RANDOM_SEED      = 42
TRAIN_SAMPLE_N   = 60_000   # target train sample size
# val sample proportional: 60000 * (5/90) ≈ 3333
VAL_SAMPLE_N     = 5000
TOP_N            = 50       # top & bottom N rows by PRODUCT_LENGTH to keep

# Load Data

In [18]:
print("Loading data with Polars (lazy scan)...")

df = pl.scan_csv(DATA_PATH).collect()

print(f"Total rows loaded: {len(df):,}")
print(f"Columns: {df.columns}")
print(f"Memory usage: ~{df.estimated_size('mb'):.1f} MB")

Loading data with Polars (lazy scan)...
Total rows loaded: 2,173,199
Columns: ['PRODUCT_ID', 'PRODUCT_TYPE_ID', 'PRODUCT_LENGTH', 'TOTAL_SENTENCE']
Memory usage: ~1481.0 MB



## 90:5:5 RANDOM SPLIT


In [19]:
print("\nSplitting 90:5:5 ...")

# Shuffle and add row index for deterministic splitting
df = df.sample(fraction=1.0, shuffle=True, seed=RANDOM_SEED)

n = len(df)
n_train = int(n * 0.90)
n_val   = int(n * 0.05)
# rest goes to test

train_full = df[:n_train]
val_full   = df[n_train : n_train + n_val]
test_full  = df[n_train + n_val :]

print(f"  train_full : {len(train_full):,} rows")
print(f"  val_full   : {len(val_full):,} rows")
print(f"  test_full  : {len(test_full):,} rows")



Splitting 90:5:5 ...
  train_full : 1,955,879 rows
  val_full   : 108,659 rows
  test_full  : 108,661 rows


## EXTRACT TOP-50 / BOTTOM-50 FROM TRAIN


In [20]:
print(f"\nExtracting top-{TOP_N} and bottom-{TOP_N} PRODUCT_LENGTH rows from train ...")

top_50    = train_full.sort("PRODUCT_LENGTH", descending=True).head(TOP_N)
bottom_50 = train_full.sort("PRODUCT_LENGTH", descending=False).head(TOP_N)

# Combine and deduplicate these anchor rows
anchor_rows = pl.concat([top_50, bottom_50]).unique()
anchor_ids  = set(anchor_rows.to_series(0).to_list())  # assumes first col is unique ID

print(f"  Anchor rows (top+bottom, deduped): {len(anchor_rows)}")


Extracting top-50 and bottom-50 PRODUCT_LENGTH rows from train ...
  Anchor rows (top+bottom, deduped): 100


## SAMPLE TRAIN SET  (anchor rows + random fill)

In [21]:


remaining_n = TRAIN_SAMPLE_N - len(anchor_rows)

# Rows in train_full that are NOT anchor rows
# Using the first column as identifier
id_col = df.columns[0]
train_remaining = train_full.filter(
    ~pl.col(id_col).is_in(anchor_rows[id_col].to_list())
)

random_fill = train_remaining.sample(n=remaining_n, seed=RANDOM_SEED, shuffle=True)

train_sample = pl.concat([anchor_rows, random_fill]).sample(
    fraction=1.0, shuffle=True, seed=RANDOM_SEED
)

print(f"\nFinal train_sample : {len(train_sample):,} rows")



Final train_sample : 60,000 rows


## SAMPLE VAL SET

In [22]:
val_sample = val_full.sample(n=min(VAL_SAMPLE_N, len(val_full)), seed=RANDOM_SEED)

print(f"Final val_sample   : {len(val_sample):,} rows")
print(f"test_full          : {len(test_full):,} rows  (full, use for final eval)")

Final val_sample   : 5,000 rows
test_full          : 108,661 rows  (full, use for final eval)


## QUICK SANITY CHECK

In [23]:
print("\n── PRODUCT_LENGTH stats in train_sample ──")
print(train_sample["PRODUCT_LENGTH"].describe())

print("\n── PRODUCT_LENGTH stats in val_sample ──")
print(val_sample["PRODUCT_LENGTH"].describe())


── PRODUCT_LENGTH stats in train_sample ──
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 60000.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 849.337277  │
│ std        ┆ 681.342055  │
│ min        ┆ 1.0         │
│ 25%        ┆ 500.0       │
│ 50%        ┆ 629.921259  │
│ 75%        ┆ 1000.0      │
│ max        ┆ 4999.999995 │
└────────────┴─────────────┘

── PRODUCT_LENGTH stats in val_sample ──
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 5000.0      │
│ null_count ┆ 0.0         │
│ mean       ┆ 846.123716  │
│ std        ┆ 660.176821  │
│ min        ┆ 1.0         │
│ 25%        ┆ 510.0       │
│ 50%        ┆ 635.0       │
│ 75%        ┆ 1000.0      │
│ max        ┆ 4921.259837 │
└────────────┴─────────────┘


# Save the Files

In [24]:
print("\nSaving splits as parquet ...")
train_sample.write_parquet("../data/train_sample.parquet")
val_sample.write_parquet("../data/val_sample.parquet")
test_full.write_parquet("../data/test_full.parquet")

print("Done! Files saved:")
print("  ../data/train_sample.parquet")
print("  ../data/val_sample.parquet")
print("  ../data/test_full.parquet")


Saving splits as parquet ...
Done! Files saved:
  ../data/train_sample.parquet
  ../data/val_sample.parquet
  ../data/test_full.parquet
